# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides users through loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL for transparent, machine-readable FAIRness.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print("Dataset Name:", metadata.get('name', ''))
print("\nDescription:", metadata.get('description', ''))

# Print main keys available in the metadata
print("\nMetadata keys:", list(metadata.keys()))

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s. The `mlcroissant` metadata provides a schema indicating the structure of the data.

Below, we enumerate record sets, their fields, and their columns using their `@id`s. This is crucial for precise referencing and programmatic data extraction.

In [ ]:
# List all record sets with their @id
record_sets = []
if 'recordSet' in metadata:
    for recordset in metadata['recordSet']:
        rec_id = recordset.get('@id', 'unknown')
        rec_name = recordset.get('name', '')
        print(f'RecordSet: @id={rec_id}, name={rec_name}')
        record_sets.append(rec_id)
        # List fields
        fields = recordset.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        for field in fields:
            fid = field.get('@id', 'unknown')
            fname = field.get('name', '')
            ftype = field.get('dataType', '')
            print(f'  Field: @id={fid}, name={fname}, dataType={ftype}')
            # List columns (if present)
            if 'column' in field:
                columns = field['column']
                if isinstance(columns, dict):
                    columns = [columns]
                for col in columns:
                    cid = col.get('@id', 'unknown')
                    cname = col.get('name', '')
                    print(f'    Column: @id={cid}, name={cname}')
else:
    print('No record sets found in the metadata.')

## 3. Data Extraction
Load data from specific record set(s) into a DataFrame for analysis. Refer to the record set and field `@id`s from the overview for precision.

Below, we simply extract *all* available record sets, if any, into dataframes for inspection.

In [ ]:
# Automatically collect all record set ids
dataframes = {}
if 'recordSet' in metadata and len(metadata['recordSet']) > 0:
    record_sets_ids = [rs['@id'] for rs in metadata['recordSet']]
    for record_set_id in record_sets_ids:
        # Extract records using the record set @id
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from {record_set_id}")
else:
    print("No record sets defined in the dataset metadata.")

# Display columns of the first record set
if dataframes:
    first_rs = list(dataframes.keys())[0]
    print(f"\nColumns in record set {first_rs}:")
    print(dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())
else:
    print("No dataframes created. Nothing to display.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Operations here include removing outliers, transforming data distributions, or grouping data by key attributes to prepare the data for further analysis.

In [ ]:
import numpy as np

# Choose the first available record set for demonstration
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id].copy()
    print(f'Working with record set: {record_set_id}')

    # Select a numeric field for analysis. For demonstration, try common mass spectrometry names:
    # Try field names: 'MW', 'coverage', 'PSM_Count', fallback to any available numeric column
    preferred_numeric_fields = ['MW', 'coverage', 'PSM_Count', 'Peptide_Count', 'Abundance', 'pI']
    numeric_field = None
    for col in df.columns:
        if col in preferred_numeric_fields and np.issubdtype(df[col].dtype, np.number):
            numeric_field = col
            break
    if not numeric_field:
        # Fallback: pick the first float or int column
        for col in df.columns:
            if np.issubdtype(df[col].dtype, np.number):
                numeric_field = col
                break
    if numeric_field:
        threshold = df[numeric_field].mean() if df[numeric_field].dtype != object else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f'{numeric_field}_normalized']].head())

        # Group by a likely categorical field
        group_fields = ['description', 'Gene', 'protein', 'accession']
        group_field = None
        for col in df.columns:
            if col in group_fields:
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"Grouped data by {group_field}:")
            display(grouped_df.head())
    else:
        print("No numeric field found for analysis.")
else:
    print("No dataframes loaded; cannot perform EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We plot the distribution of the selected numeric field and the relationship between groups when possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    if group_field:
        plt.figure(figsize=(10, 6))
        top_groups = df[group_field].value_counts().index[:10]
        sns.boxplot(data=df[df[group_field].isin(top_groups)], x=group_field, y=numeric_field)
        plt.title(f'{numeric_field} by {group_field} (top 10 groups)')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("Nothing to visualize.")

## 6. Conclusion
In this notebook, we demonstrated how to use `mlcroissant` to programmatically explore the FAIR² dataset defined by a Croissant JSON-LD schema. We examined the schema structure (record sets and fields), loaded and processed the data, applied basic exploratory analysis, and visualized important protein metrics.

This approach ensures reproducibility and transparency for protein mass spectrometry datasets and facilitates easy adoption and further analysis for the research community.